# 06 · Fortran extras, makefiles and column lists

Inlists are merged key by key. The other files a run needs are **used whole**: the Fortran sources (`run_star_extras.f`, `run_binary_extras.f`, `run.f`, `binary_run.f`), the makefiles, and the three column lists. This is `posydon/grids/setup/extras.py`:

| Step | Function |
|---|---|
| combine the ini's extras with the scenario's | `merge_extras` |
| one file per place: the highest layer wins | `keep_highest_extras` |
| where each key's file goes | `destinations` |
| copy them and write `mk` | `stage_extras`, `mk_lines` |
| compile | `compile_executables` |
| pick, check and copy the column lists | `column_list_sources`, `replaced_column_lists`, `stage_column_lists` |

The keys keep their legacy `[mesa_extras]` names (`mesa_star1_extras`, `posydon_binary_extras`, …): the prefix before the first `_` is the layer (`mesa` < `posydon` < `user`), the rest says where the file goes.

In [ ]:
from nb_env import env, fresh, write, show, short, grid_csv, tree
ENV = env()

import os
from pathlib import Path
import pandas as pd
from posydon.grids.setup.config import read_config
from posydon.grids.setup.resolve import resolve_scenario
from posydon.grids.setup.defaults import fill_defaults
from posydon.grids.setup.extras import (
    compile_executables, column_list_sources, destinations, keep_highest_extras,
    merge_extras, mk_lines, replaced_column_lists, stage_column_lists, stage_extras)

def rel(path):
    return short(path, ENV.release, ENV.inlists / "r11701", ENV.mesa, ENV.work) if path else path

folder = fresh("06_extras")
grid_csv(folder)
config = read_config(write(folder / "grid.ini", f"""
    [grid]
    type = CO-HeMS
    points = grid.csv
    [scenario]
    release = v2.0
    inlist_repo = {ENV.inlists}
    """))
resolved = resolve_scenario(config)
fill_defaults(config, resolved, os.environ["MESA_DIR"])
scenario = resolved.files

## Where each key's file goes

In [2]:
keys = ["makefile_binary", "makefile_star", "binary_run", "star_run", "mesa_binary_extras",
        "posydon_star_binary_extras", "mesa_star1_extras", "user_star2_extras", "mesa_dir", "something_else"]
pd.DataFrame({"goes to": [", ".join(destinations(k)) or "(not a file)" for k in keys]}, index=keys)

,goes to
makefile_binary,binary/make/makefile_binary
makefile_star,"star1/make/makefile_star, star2/make/makefile_..."
binary_run,binary/src/
star_run,"star1/src/, star2/src/"
mesa_binary_extras,binary/src/
posydon_star_binary_extras,binary/src/
mesa_star1_extras,star1/src/
user_star2_extras,star2/src/
mesa_dir,(not a file)
something_else,(not a file)


## The ini's extras and the scenario's

The ini's extras (here: MESA's stock sources, filled in by `fill_defaults`) and the scenario's are combined. A scenario key replaces an ini key of the same name.

In [3]:
merged = merge_extras(config.extras, scenario.extras)
pd.DataFrame({"ini": {k: rel(v) for k, v in config.extras.items()},
              "scenario": {k: rel(v) for k, v in scenario.extras.items()},
              "merged": {k: rel(v) for k, v in merged.items()}}).fillna("")

,ini,scenario,merged
makefile_binary,binary/work/make/makefile,,binary/work/make/makefile
makefile_star,star/work/make/makefile,,star/work/make/makefile
mesa_binary_extras,binary/work/src/run_binary_extras.f,,binary/work/src/run_binary_extras.f
mesa_star_binary_extras,binary/work/src/run_star_extras.f,,binary/work/src/run_star_extras.f
mesa_star1_extras,star/work/src/run_star_extras.f,common/binary/src/run_star_extras.f,common/binary/src/run_star_extras.f
mesa_star2_extras,star/work/src/run_star_extras.f,,common/binary/src/run_star_extras.f
binary_run,binary/work/src/binary_run.f,,binary/work/src/binary_run.f
star_run,star/work/src/run.f,,star/work/src/run.f
posydon_binary_extras,,common/binary/src/run_binary_extras.f,common/binary/src/run_binary_extras.f
posydon_star_binary_extras,,common/binary/src/run_star_extras.f,common/binary/src/run_star_extras.f


### Fixed C1: star 2 gets the scenario's `run_star_extras.f` too

If the scenario gives star 1 its `run_star_extras.f` but names none for star 2, star 2 gets the same file. Before, star 2 kept MESA's stock one from the ini.

In [4]:
merge_extras({"mesa_star1_extras": "MESA/run_star_extras.f", "mesa_star2_extras": "MESA/run_star_extras.f"},
             {"mesa_star1_extras": "POSYDON/run_star_extras.f"})

{'mesa_star1_extras': PosixPath('POSYDON/run_star_extras.f'),
 'mesa_star2_extras': PosixPath('POSYDON/run_star_extras.f')}

## One file per place: the highest layer wins

Several keys can put a file with the same name in the same place. `mesa_binary_extras` and `posydon_binary_extras` both write `binary/src/run_binary_extras.f`. Only the highest layer's key is kept: `mesa` < `posydon` < `user`. Keys without a layer prefix (`binary_run`, `star_run`, makefiles) are always kept.

In [5]:
kept = keep_highest_extras(merged)
dropped = [k for k in merged if k not in kept]
print("dropped:", dropped)
pd.DataFrame({"file": {k: rel(v) for k, v in kept.items()},
              "goes to": {k: ", ".join(destinations(k)) for k in kept}})

dropped: ['mesa_binary_extras', 'mesa_star_binary_extras']


,file,goes to
makefile_binary,binary/work/make/makefile,binary/make/makefile_binary
makefile_star,star/work/make/makefile,"star1/make/makefile_star, star2/make/makefile_..."
mesa_star1_extras,common/binary/src/run_star_extras.f,star1/src/
mesa_star2_extras,common/binary/src/run_star_extras.f,star2/src/
binary_run,binary/work/src/binary_run.f,binary/src/
star_run,star/work/src/run.f,"star1/src/, star2/src/"
posydon_binary_extras,common/binary/src/run_binary_extras.f,binary/src/
posydon_star_binary_extras,common/binary/src/run_star_extras.f,binary/src/


Here the dropped keys are MESA's stock files that `fill_defaults` filled in, and losing to POSYDON's is what they are for, so the setup says nothing. A key you wrote yourself in `[extras]` that loses is reported (`[extras] mesa_binary_extras is not used: a higher layer's file goes to the same place`); a legacy ini keeps today's `Section mesa_extras value ... is being set to None`.

### Fixed C4: decided by layer, not by the alphabet

Before, the alphabetically last prefix of all `*binary_extras*` keys won, and every other `*binary_extras*` key was dropped, whatever file it held. A `user_` file now beats a `posydon_` one, and keys for different files no longer knock each other out:

In [6]:
keep_highest_extras({"mesa_binary_extras": "MESA/run_binary_extras.f",
                     "posydon_binary_extras": "POSYDON/run_binary_extras.f",
                     "user_binary_extras": "MINE/run_binary_extras.f",
                     "posydon_star_binary_extras": "POSYDON/run_star_extras.f"})

{'user_binary_extras': 'MINE/run_binary_extras.f',
 'posydon_star_binary_extras': 'POSYDON/run_star_extras.f'}

## Copy them, and `mk`

`stage_extras` empties `binary/src`, `binary/make`, `star1/...`, `star2/...`, copies the files and writes `mk`: one `cd` + `make -f` per makefile. `mk` starts with `#!/bin/sh`, so you can rerun it by hand (fixed C15).

In [7]:
run = folder / "run"
mk = stage_extras(kept, run)
tree(run, skip=())
show(mk)

binary/make/makefile_binary
binary/src/binary_run.f
binary/src/run_binary_extras.f
binary/src/run_star_extras.f
mk
star1/make/makefile_star
star1/src/run.f
star1/src/run_star_extras.f
star2/make/makefile_star
star2/src/run.f
star2/src/run_star_extras.f
#!/bin/sh
cd /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/06_extras/run/binary/make
make -f makefile_binary
cd /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/06_extras/run/star1/make
make -f makefile_star
cd /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/06_extras/run/star2/make
make -f makefile_star


Which file ended up where (matched by content):

In [8]:
for path in sorted(run.rglob("*.f")):
    origin = [rel(src) for src in kept.values() if Path(src).read_bytes() == path.read_bytes()]
    print(f"{str(path.relative_to(run)):32} <- {origin[0]}")

binary/src/binary_run.f          <- binary/work/src/binary_run.f
binary/src/run_binary_extras.f   <- common/binary/src/run_binary_extras.f
binary/src/run_star_extras.f     <- common/binary/src/run_star_extras.f
star1/src/run.f                  <- star/work/src/run.f
star1/src/run_star_extras.f      <- common/binary/src/run_star_extras.f
star2/src/run.f                  <- star/work/src/run.f
star2/src/run_star_extras.f      <- common/binary/src/run_star_extras.f


`compile_executables` runs `sh ./mk` in the run directory and returns the three executable paths the run needs. The stub makefiles do nothing, so the files do not exist here.

In [9]:
executables = compile_executables(run)
[(rel(e), Path(e).exists()) for e in executables]

[('06_extras/run/binary/binary', False),
 ('06_extras/run/star1/star', False),
 ('06_extras/run/star2/star', False)]

## Your own Fortran source

Put it in `[user_inlists]` as a whole file, like a column list. `run_star_extras` is used wherever a star runs (`binary/src`, `star1/src`, `star2/src`); `run_binary_extras` in `binary/src`. `whole_file_extras` turns them into `user_*` extras keys, which beat POSYDON's. The file can have any name: sources are copied under the name MESA compiles (`file_name`).

Before, these two keys were accepted and then ignored; a source in `[extras]` was copied under its own name (which MESA never compiles), and naming any `[extras]` key stopped MESA's makefiles from being filled in, so nothing was compiled.

In [10]:
from posydon.grids.setup.extras import file_name, whole_file_extras
from posydon.grids.setup.steps import chosen_extras

mine = write(folder / "my_star_extras.f", "! MY run_star_extras\n")
config.inlists["user_inlists"].whole_files["run_star_extras"] = mine
print({k: rel(v) for k, v in whole_file_extras(config).items()})
print(file_name("user_star1_extras", mine))

run_mine = folder / "run_mine"
stage_extras(chosen_extras(config, scenario), run_mine)
for path in sorted(run_mine.rglob("run_star_extras.f")):
    print(f"{str(path.relative_to(run_mine)):28} {path.read_text().strip()[:40]}")

{'user_star_binary_extras': '06_extras/my_star_extras.f', 'user_star1_extras': '06_extras/my_star_extras.f', 'user_star2_extras': '06_extras/my_star_extras.f'}
run_star_extras.f
binary/src/run_star_extras.f ! MY run_star_extras
star1/src/run_star_extras.f  ! MY run_star_extras
star2/src/run_star_extras.f  ! MY run_star_extras


## Column lists

The three column lists (`history_columns`, `binary_history_columns`, `profile_columns`) go through `pick_whole_file`: the highest layer that has one wins. Within `posydon_inlists`, the scenario's list replaces the ini's.

In [11]:
{k: rel(v) for k, v in column_list_sources(config, scenario).items()}

{'history_columns': 'common/history_columns.list',
 'binary_history_columns': 'common/binary_history_columns.list',
 'profile_columns': 'common/profile_columns.list'}

### Fixed C7: say when the scenario replaces the ini's lists

A legacy ini that names its own column lists (as every one does; they go to its `posydon_inlists` layer) still gets the scenario's. That was silent; now `replaced_column_lists` names the lists, and the setup warns. A list with the same content is not reported.

In [12]:
mesa_columns = ENV.inlists / "r11701" / "mesa" / "default" / "columns"
config.inlists["posydon_inlists"].whole_files["history_columns"] = mesa_columns / "history_columns.list"
config.inlists["posydon_inlists"].whole_files["profile_columns"] = ENV.release / "common" / "profile_columns.list"  # same file
print("replaced:", replaced_column_lists(config, scenario))

replaced: ['history_columns']


To really use your own list, put it in `[user_inlists]`: the user layer beats both.

In [13]:
mine = write(folder / "my_history_columns.list", "star_age\nstar_mass\nlog_L\nlog_Teff\n")
config.inlists["user_inlists"].whole_files["history_columns"] = mine
sources = column_list_sources(config, scenario)
{k: rel(v) for k, v in sources.items()}

{'history_columns': '06_extras/my_history_columns.list',
 'binary_history_columns': 'common/binary_history_columns.list',
 'profile_columns': 'common/profile_columns.list'}

In [14]:
staged = stage_column_lists(sources, run)
{k: rel(v) for k, v in staged.items()}

{'history_columns': '06_extras/run/column_lists/history_columns.list',
 'binary_history_columns': '06_extras/run/column_lists/binary_history_columns.list',
 'profile_columns': '06_extras/run/column_lists/profile_columns.list'}

## Try it

- Add `[extras] user_star1_extras = my_run_star_extras.f` to an ini and see where it lands and what it beats.
- Pass a rerun (`rerun = matthias_lbv_thermo_dedt_noAgeLimit`) and check which sources it replaces (`[replace]` in its `scenario.ini`).